# B4 — Clasificador de Contratación Pública

Notebook de clasificación taxonómica para publicaciones del dominio de contratación pública.
Sigue la misma arquitectura incremental que B0-B3.

**Particularidades de B4 frente a bloques anteriores:**
- **Sesgo BOE crítico**: ~59% de la contratación está en BOE (la autonómica y local publica en plataformas de contratación, no en boletines). Documentar en el análisis.
- CONT_CON es un **tipo de contrato**, no una fase: se combina con CONT_LIC/CONT_FOR/CONT_ADJ.
- Fronteras duras: "se adjudica" casi siempre es RRHH (~780 registros de plazas/puestos); "concesión administrativa/demanial" es dominio público (~226 registros, dominio B1).

In [ ]:
import os
import html
import re
import json
import asyncio
from pathlib import Path

import pandas as pd
import numpy as np

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    hamming_loss,
    jaccard_score,
)
from sklearn.preprocessing import MultiLabelBinarizer

from dotenv import find_dotenv, load_dotenv
from pydantic_ai import Agent

SEED = 29092025

load_dotenv(find_dotenv())

In [ ]:
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider

LM_STUDIO_MODEL = "qwen/qwen3.5-9b"

model = OpenAIChatModel(
    LM_STUDIO_MODEL,
    provider=OpenAIProvider(
        base_url="http://localhost:1234/v1",
        api_key="lm-studio",
    ),
)

In [ ]:
import httpx

try:
    r = httpx.get("http://localhost:1234/v1/models", timeout=3)
    modelos = [m["id"] for m in r.json()["data"]]
    assert LM_STUDIO_MODEL in modelos, f"{LM_STUDIO_MODEL} no está cargado en LM Studio"
    print(f"OK: {LM_STUDIO_MODEL} listo  |  otros: {[m for m in modelos if m != LM_STUDIO_MODEL]}")
except httpx.ConnectError:
    raise RuntimeError("LM Studio no responde - arrancar el servidor antes de continuar")

## 1. Schema B4

In [ ]:
from clasificador.schema_B4 import (
    ActType, CategoryTypeB4, SubcategoryTypeB4, ClassifierOutputB4
)

In [ ]:
ejemplo_valido = ClassifierOutputB4(
    is_relevant=True,
    act_type=ActType.ANUNCIO,
    categories=[CategoryTypeB4.CONT_LIC, CategoryTypeB4.CONT_CON],
    subcategories=[SubcategoryTypeB4.TIPO_SERVICIOS],
    confidence=0.95,
    reasoning="'Anuncio de licitación' → CONT_LIC. 'Objeto: Concesión de servicios para la explotación de cafetería' → CONT_CON + tipo_servicios.",
)
print("Ejemplo válido:")
print(ejemplo_valido.model_dump_json(indent=2))

print("\nViolación de invariante:")
try:
    ClassifierOutputB4(
        is_relevant=False, act_type=ActType.ANUNCIO,
        categories=[CategoryTypeB4.CONT_FOR], subcategories=[],
        confidence=0.5, reasoning="Prueba.",
    )
except Exception as e:
    print(f"  ValidationError -> {e.errors()[0]['msg']}")

## 2. Exploración del corpus B4

In [ ]:
from clasificador.agent import get_ambito, inferir_act_type

PATH_PARQUET = "../data/raw/silver_official_gazettes_2025_Q1.parquet"
df = pd.read_parquet(PATH_PARQUET)
df["description"] = df["description"].apply(lambda x: html.unescape(str(x)) if pd.notna(x) else x)
print(f"Corpus total: {len(df):,} registros | Columnas: {list(df.columns)}")

In [ ]:
df["act_type_n1"] = df.apply(
    lambda r: inferir_act_type(str(r["description"]), str(r["bulletin"])).value, axis=1
)

keywords_dominio_B4 = [
    "anuncio de licitación", "licitación de", "licitación para",
    "pliego de cláusulas", "pliego de condiciones",
    "formalización del contrato", "formalización de contrato", "formalización contrato",
    "adjudicación del contrato", "adjudicación de contrato",
    "encargo a medio propio", "encomienda de gestión",
    "concesión de servicio", "concesión del servicio", "contrato de concesión",
]
desc_lower = df["description"].str.lower()
mask_b4 = desc_lower.str.contains("|".join(keywords_dominio_B4), na=False)
df_b4 = df[mask_b4].copy()
print(f"Universo B4 estimado: {len(df_b4):,} registros ({len(df_b4)/len(df)*100:.2f}% del corpus)")
print(f"\nDistribución por boletín (top 10) - NOTA: sesgo BOE esperado ~59%:")
print(df_b4["bulletin"].value_counts().head(10).to_string())
print(f"\nDistribución N1 en universo B4:")
print(df_b4["act_type_n1"].value_counts().head(8).to_string())

## 3. Ground truth — Muestreo estratificado

Cuotas por grupo (total: 150):

| Grupo | Cuota | Notas |
|-------|-------|-------|
| CONT_FOR | 30 | formalizaciones (la etiqueta dominante del corpus) |
| CONT_LIC | 30 | licitaciones y pliegos |
| CONT_ADJ | 15 | pool pequeño (~60 con keywords estrictas) |
| CONT_ENC | 15 | encargos y encomiendas |
| CONT_CON | 15 | concesiones de servicios/obras (genera multilabel con LIC/FOR) |
| NEG_ADJ_RRHH | 10 | adjudicaciones de plazas/puestos (frontera B5) |
| NEG_DOM_PUB | 10 | concesiones demaniales/administrativas (frontera B1) |
| NEGATIVO | 25 | resto del corpus sin señal B4 |

In [ ]:
keywords_B4 = {
    "CONT_LIC": ["anuncio de licitación", "licitación de", "licitación para",
                 "convocatoria de licitación", "pliego de cláusulas",
                 "pliego de condiciones"],
    "CONT_FOR": ["formalización del contrato", "formalización de contrato",
                 "formalización contrato"],
    "CONT_ADJ": ["adjudicación del contrato", "adjudicación de contrato",
                 "adjudicación contrato", "se adjudica el contrato"],
    "CONT_ENC": ["encargo a medio propio", "encargo al medio propio",
                 "encomienda de gestión", "como medio propio"],
    "CONT_CON": ["concesión de servicio", "concesión del servicio",
                 "concesión de obra", "contrato de concesión"],
}

cuotas_B4 = {"CONT_FOR": 30, "CONT_LIC": 30, "CONT_ADJ": 15, "CONT_ENC": 15, "CONT_CON": 15}
N_NEG_ADJ_RRHH = 10
N_NEG_DOM_PUB  = 10
N_NEGATIVOS    = 25

desc_lower = df["description"].str.lower()

def mask_kw(kws):
    return desc_lower.str.contains("|".join(kws), na=False)

# Negativos difíciles
mask_adj_rrhh = (
    (desc_lower.str.contains("se adjudica", na=False) | desc_lower.str.contains("adjudicación de", na=False))
    & desc_lower.str.contains("plaza|puesto|destino", na=False)
    & ~mask_kw(keywords_B4["CONT_ADJ"])
)
mask_dom_pub = (
    desc_lower.str.contains("concesión administrativa|concesión demanial", na=False)
    & ~mask_kw(keywords_B4["CONT_CON"])
)

print(f"{'Label':<14} {'Pool':>8} {'Cuota':>7} {'Estado':>14}")
print("-" * 48)
for label, kws in keywords_B4.items():
    pool = mask_kw(kws).sum()
    cuota = cuotas_B4.get(label, 0)
    estado = "OK" if pool >= cuota else f"REDUCIDA a {min(cuota, pool)}"
    print(f"{label:<14} {pool:>8,} {cuota:>7} {estado:>14}")
print(f"{'NEG_ADJ_RRHH':<14} {mask_adj_rrhh.sum():>8,} {N_NEG_ADJ_RRHH:>7}")
print(f"{'NEG_DOM_PUB':<14} {mask_dom_pub.sum():>8,} {N_NEG_DOM_PUB:>7}")

In [ ]:
sampled_ids = set()
frames = []

def tomar_muestra(pool_df, n, grupo):
    n = min(n, len(pool_df))
    if n == 0:
        print(f"  {grupo:<14} SKIP (pool vacío)")
        return
    sample = pool_df.sample(n, random_state=SEED).copy()
    sample["grupo_muestreo"] = grupo
    sampled_ids.update(sample.index.tolist())
    frames.append(sample)
    print(f"  {grupo:<14} pool={len(pool_df):>5,}  sampled={n}")

# 1. Pools pequeños primero
for label in ["CONT_CON", "CONT_ADJ", "CONT_ENC"]:
    mask = mask_kw(keywords_B4[label]) & ~df.index.isin(sampled_ids)
    tomar_muestra(df[mask], cuotas_B4[label], label)

# 2. Negativos difíciles
mask = mask_adj_rrhh & ~df.index.isin(sampled_ids)
tomar_muestra(df[mask], N_NEG_ADJ_RRHH, "NEG_ADJ_RRHH")
mask = mask_dom_pub & ~df.index.isin(sampled_ids)
tomar_muestra(df[mask], N_NEG_DOM_PUB, "NEG_DOM_PUB")

# 3. Pools grandes
for label in ["CONT_LIC", "CONT_FOR"]:
    mask = mask_kw(keywords_B4[label]) & ~df.index.isin(sampled_ids)
    tomar_muestra(df[mask], cuotas_B4[label], label)

# 4. Negativos del resto del corpus
all_kws_b4 = [kw for kws in keywords_B4.values() for kw in kws]
mask_neg = (~desc_lower.str.contains("|".join(all_kws_b4), na=False)
            & ~df.index.isin(sampled_ids))
tomar_muestra(df[mask_neg], N_NEGATIVOS, "NEGATIVO")

df_muestreo = pd.concat(frames, ignore_index=True)
df_muestreo["id"] = range(len(df_muestreo))
print(f"\nTotal muestreado: {len(df_muestreo)} registros")
print(df_muestreo["grupo_muestreo"].value_counts().to_string())
print(f"\nBOE en la muestra: {(df_muestreo['bulletin'] == 'boe').sum()}/{len(df_muestreo)}")

In [ ]:
print("Validación del muestreo - 3 ejemplos por grupo:\n")
for grupo in df_muestreo["grupo_muestreo"].unique():
    muestra = df_muestreo[df_muestreo["grupo_muestreo"] == grupo].head(3)
    print(f"--- {grupo} ---")
    for _, row in muestra.iterrows():
        desc = str(row["description"])[:130].replace("\n", " ")
        bul  = str(row.get("bulletin", "?")).upper()
        print(f"  [{bul}] {desc}")
    print()

In [ ]:
PATH_MUESTREO = "../data/ground_truth/ground_truth_B4_muestreo.csv"
Path(PATH_MUESTREO).parent.mkdir(parents=True, exist_ok=True)

df_muestreo["is_relevant_gt"]   = ""
df_muestreo["categories_gt"]    = ""
df_muestreo["subcategories_gt"] = ""
df_muestreo["notas_anotador"]   = ""

df_muestreo[["id","bulletin","description","grupo_muestreo",
             "is_relevant_gt","categories_gt","subcategories_gt","notas_anotador"]].to_csv(
    PATH_MUESTREO, index=False
)
print(f"Guardado: {PATH_MUESTREO}  ({len(df_muestreo)} registros)")
print("Siguiente paso: anotar manualmente las columnas is_relevant_gt, categories_gt, subcategories_gt")

## 4. Agente base

In [ ]:
from clasificador.schema_B4 import ClassifierOutputB4
from clasificador.prompts_B4 import PROMPT_REGISTRY_B4
from clasificador.agent import build_agent, run_experiment

In [ ]:
agent_b4_v1 = build_agent(
    model, "v1",
    output_type=ClassifierOutputB4,
    prompt_registry=PROMPT_REGISTRY_B4,
)

# Casos cualitativos representativos del dominio B4
casos_b4 = [
    ("Anuncio de licitación de: Dirección Provincial del Ministerio de Educación en Melilla. Objeto: Limpieza de 6 Centros Educativos. Expediente: 2024/003.", "boe"),
    ("Anuncio de formalización de contratos de: Jefatura de Asuntos Económicos de la Guardia Civil. Objeto: Suministro de munición para el ejercicio 2025.", "boe"),
    ("Anuncio de licitación de: Subsecretaría de Política Territorial. Objeto: Concesión de servicios para la instalación, explotación y mantenimiento de la cafetería del edificio.", "boe"),
    ("RESOLUCIÓN de 13 de enero de 2025, de la Dirección General de Movilidad, por la que se dispone la publicación de la prórroga del Convenio de encomienda de gestión a TRAGSA para la explotación de estaciones de autobuses.", "doe"),
    ("Resolución de 27 de diciembre de 2024, de la Viceconsejería, por la que se adjudica puesto de trabajo de libre designación convocado por resolución de 30 de octubre.", "bocm"),
]

for desc, bul in casos_b4:
    print(f"\n[{bul.upper()}] {desc[:80]}...")
    # result = await agent_b4_v1.run(f"Boletín: {bul.upper()}\n\nDescripción: {desc}")
    # print(result.output.model_dump_json(indent=2))

## 5. Funciones de evaluación B4

In [ ]:
def parse_labels(value) -> set:
    """Convierte cualquier representación de etiquetas a un set de strings."""
    if pd.isna(value) or str(value).strip() in ("", "nan"):
        return set()
    s = str(value).strip()
    if s.startswith("["):
        try:
            items = json.loads(s)
            return {str(i).strip('"') for i in items if i}
        except json.JSONDecodeError:
            pass
    return {v.strip().strip('"') for v in s.split(",") if v.strip()}

In [ ]:
def compute_metrics_B4(df_eval, label="", verbose=True):
    """
    Calcula métricas multilabel para el clasificador B4.
    Columnas esperadas: is_relevant_gt, categories_gt, is_relevant_pred, categories_pred.
    """
    ALL_CATS_B4 = [e.value for e in CategoryTypeB4]
    mlb = MultiLabelBinarizer(classes=ALL_CATS_B4)
    mlb.fit([ALL_CATS_B4])

    gt_labels   = [parse_labels(v) & set(ALL_CATS_B4) for v in df_eval["categories_gt"]]
    pred_labels = [parse_labels(v) & set(ALL_CATS_B4) for v in df_eval["categories_pred"]]
    Y    = mlb.transform(gt_labels)
    Yhat = mlb.transform(pred_labels)

    is_rel_gt   = df_eval["is_relevant_gt"].astype(bool)
    is_rel_pred = df_eval["is_relevant_pred"].astype(bool)

    exact = pd.Series([set(g) == set(p) for g, p in zip(gt_labels, pred_labels)])
    rel   = is_rel_gt

    metrics = {
        "is_rel_accuracy":  round(accuracy_score(is_rel_gt, is_rel_pred), 4),
        "is_rel_precision": round(precision_score(is_rel_gt, is_rel_pred, zero_division=0), 4),
        "is_rel_recall":    round(recall_score(is_rel_gt, is_rel_pred, zero_division=0), 4),
        "is_rel_f1":        round(f1_score(is_rel_gt, is_rel_pred, zero_division=0), 4),
        "micro_f1":         round(f1_score(Y, Yhat, average="micro", zero_division=0), 4),
        "macro_f1":         round(f1_score(Y, Yhat, average="macro", zero_division=0), 4),
        "hamming_loss":     round(hamming_loss(Y, Yhat), 4),
        "jaccard_samples":  round(jaccard_score(Y, Yhat, average="samples", zero_division=0), 4),
        "subset_accuracy":  round(accuracy_score(Y, Yhat), 4),
        "exact": exact,
        "rel":   rel,
    }

    if verbose:
        title = f"-- {label} --" if label else "-- Métricas B4 --"
        print(f"\n{title}\n")
        tp = int((is_rel_gt & is_rel_pred).sum())
        fp = int((~is_rel_gt & is_rel_pred).sum())
        fn = int((is_rel_gt & ~is_rel_pred).sum())
        tn = int((~is_rel_gt & ~is_rel_pred).sum())
        print(f"is_relevant  Acc={metrics['is_rel_accuracy']:.3f}  P={metrics['is_rel_precision']:.3f}  "
              f"R={metrics['is_rel_recall']:.3f}  F1={metrics['is_rel_f1']:.3f}")
        print(f"             TP={tp}  FP={fp}  FN={fn}  TN={tn}\n")
        print(f"N2 multilabel:")
        print(f"  Micro F1:      {metrics['micro_f1']:.3f}")
        print(f"  Macro F1:      {metrics['macro_f1']:.3f}")
        print(f"  Hamming Loss:  {metrics['hamming_loss']:.4f}")
        print(f"  Jaccard:       {metrics['jaccard_samples']:.3f}")
        print(f"  Subset Acc:    {metrics['subset_accuracy']:.3f}\n")
        f1s = f1_score(Y, Yhat, average=None, zero_division=0)
        sups = Y.sum(axis=0)
        print(f"  {'Label':<12} {'P':>6} {'R':>6} {'F1':>6} {'Sup':>5}")
        print(f"  {'-'*37}")
        for i, cat in enumerate(ALL_CATS_B4):
            prec = precision_score(Y[:, i], Yhat[:, i], zero_division=0)
            rec  = recall_score(Y[:, i], Yhat[:, i], zero_division=0)
            print(f"  {cat:<12} {prec:>6.3f} {rec:>6.3f} {f1s[i]:>6.3f} {int(sups[i]):>5}")
        print(f"  {'-'*37}")
        print(f"  {'Macro':<12} {'':>6} {'':>6} {metrics['macro_f1']:>6.3f}\n")

        cards = [len(g) for g in gt_labels]
        print(f"  Subset Acc por cardinalidad:")
        for card in [0, 1, 2]:
            idx = [i for i, c in enumerate(cards) if c == card]
            if idx:
                acc = accuracy_score(Y[idx], Yhat[idx])
                print(f"    card={card} ({'no relevante' if card == 0 else str(card)}) : {acc:.3f}  ({int(acc*len(idx))}/{len(idx)})")
        idx3 = [i for i, c in enumerate(cards) if c >= 3]
        if idx3:
            acc3 = accuracy_score(Y[idx3], Yhat[idx3])
            print(f"    card>=3               : {acc3:.3f}  ({int(acc3*len(idx3))}/{len(idx3)})")

        conf = df_eval.get("confidence", pd.Series(dtype=float))
        if conf.notna().any():
            print(f"\n  Confianza: media={conf.mean():.3f}  min={conf.min():.3f}  max={conf.max():.3f}")

    return metrics

In [ ]:
def print_errors_B4(df_eval, exact, rel, label="", n=None):
    errores = df_eval[~exact & rel]
    if n is not None:
        errores = errores.head(n)
    lbl = f" · {label}" if label else ""
    print(f"\n-- Errores N2 en relevantes{lbl} --")
    print(f"Total: {len(errores)}\n")
    for _, row in errores.iterrows():
        gt   = sorted(parse_labels(row["categories_gt"]))
        pred = sorted(parse_labels(row["categories_pred"]))
        falta = sorted(set(gt) - set(pred))
        sobra = sorted(set(pred) - set(gt))
        desc  = str(row["description"])[:90].replace("\n", " ")
        razon = str(row.get("reasoning", "")).replace("\n", " ")[:120]
        print(f"ID {row['id']} | GT={gt} | PRED={pred}")
        print(f"  Falta: {falta} | Sobra: {sobra}")
        print(f"  {desc}...")
        print(f"  Razonamiento: {razon}")
        print()

---

##  6. Experimento 1 - Baseline zero-shot

**Problema**: No existe un clasificador para el dominio de contratación pública. Necesitamos una línea base que mida el rendimiento zero-shot antes de cualquier optimización.

**Objetivo**: Establecer el Macro-F1 de referencia con el prompt mínimo operativo (V1) y detectar los patrones de error sistemáticos. Atención especial a: (1) frontera adjudicación de contrato vs adjudicación de plaza (RRHH), (2) frontera concesión contractual vs concesión demanial (B1), (3) multilabel CONT_CON + fase.

**Enfoque**: Qwen 3.5 9B · SYSTEM_PROMPT_B4_V1 · zero-shot · sin contexto N1.

**Resultados**: is_rel F1=0.935 (P=1.000, R=0.878, **12 FN**) · Micro F1=0.881 · **Macro F1=0.840** · Subset Acc=0.840.

| Label | P | R | F1 | Sup |
|-------|------|------|------|-----|
| CONT_LIC | 0.972 | 0.972 | 0.972 | 36 |
| CONT_FOR | 0.969 | 1.000 | 0.984 | 31 |
| CONT_ADJ | 1.000 | 0.900 | 0.947 | 10 |
| CONT_ENC | 1.000 | 0.600 | 0.750 | 15 |
| CONT_CON | 0.474 | 0.643 | **0.545** | 14 |

**Análisis de errores (24 en relevantes)**: las dos reglas frontera de V1 sobre-corrigieron y hay una confusión conceptual nueva:
1. **CONT_CON espuria (10 errores, P=0.474)**: confunde "contrato DE servicios" con "CONCESIÓN de servicios": añade CONT_CON a licitaciones y formalizaciones ordinarias de vigilancia, limpieza, soporte (ids 66, 74, 77, 91, 99, 102, 104, 105, 112, 121).
2. **Regla demanial sobre-aplicada (5 FN: ids 2, 3, 4, 7, 9)**: marca como demaniales concesiones de servicios LCSP reales (cafetería de la Fuerza Terrestre, piscinas municipales, abastecimiento de Alcantarilla, aparcamiento de Oviedo).
3. **Regla de convenios sobre-aplicada (6 FN: ids 33, 34, 38, 40, 42, 44)**: descarta los convenios BOPA que formalizan encomiendas de gestión de ayuda a domicilio (R de CONT_ENC=0.600).
4. Menores: "se formaliza la encomienda" → CONT_FOR espuria (id 41), FOR+LIC espuria (id 109), adjudicación con mención de subvenciones → [] (id 23).

Los negativos están perfectos (52/52): las fronteras RRHH, DOP y demanial-pura funcionan. El problema es que el modelo no distingue el lado relevante de cada frontera. V2 reescribe las 3 reglas con señales de ambos lados.

In [ ]:
df_anotado = pd.read_csv("../data/ground_truth/ground_truth_B4_anotado.csv")
df_anotado_run = df_anotado[df_anotado["is_relevant_gt"].notna()].copy()

df_b4_exp1 = await run_experiment(
    agent_b4_v1, df_anotado_run,
    use_n1_context=False, concurrency=1,
    output_path="../results/b4_exp1_baseline_qwen9b.csv",
    desc="B4 Exp1 - Baseline V1",
)
df_b4_exp1.head(3)

In [ ]:
df_b4_exp1 = pd.read_csv("../results/b4_exp1_baseline_qwen9b.csv")
df_eval_b4_1 = df_anotado[["id","is_relevant_gt","categories_gt","subcategories_gt","description"]].merge(
    df_b4_exp1[["description","is_relevant_pred","act_type_pred","categories_pred",
                "subcategories_pred","confidence","reasoning"]], on="description", how="left"
)
m_b4_1 = compute_metrics_B4(df_eval_b4_1, "Experimento B4-1 - Baseline")
print_errors_B4(df_eval_b4_1, m_b4_1["exact"], m_b4_1["rel"], label="B4 Experimento 1 - Baseline")

---

##  7. Experimento 2 - Prompt v2 (correcciones post-baseline)

**Problema**: El baseline (Macro F1=0.840) tiene precisión 1.000 en negativos pero las reglas frontera sobre-corrigieron hacia dentro: CONT_CON espuria en contratos ordinarios de servicios (P=0.474), 5 concesiones LCSP reales descartadas como demaniales y 6 encomiendas descartadas como convenios interadministrativos (R de CONT_ENC=0.600).

**Objetivo**: Verificar si V2 corrige los 3 patrones con reglas bidireccionales: (1) "contrato DE servicios ≠ CONCESIÓN de servicios" (CONT_CON exige la palabra concesión en el objeto), (2) frontera demanial/LCSP con señales de ambos lados (ocupación de dominio público vs concesión de servicios de cafetería/piscinas/agua/transporte), (3) el convenio que formaliza una encomienda ES CONT_ENC, y los actos preparatorios de una concesión (viabilidad, estructura de costes) son CONT_CON.

**Enfoque**: Qwen 3.5 9B · SYSTEM_PROMPT_B4_V2 (4.066 chars) · zero-shot · sin contexto N1.

**Resultados**: is_rel F1=0.964 · Micro F1=0.967 · **Macro F1=0.951** (+0.111 vs baseline) · Subset Acc=0.953 (+0.113).

| Label | P | R | F1 | Δ F1 vs Exp1 |
|-------|------|------|------|------|
| CONT_LIC | 1.000 | 1.000 | 1.000 | +0.028 |
| CONT_FOR | 1.000 | 0.935 | 0.967 | -0.017 |
| CONT_ADJ | 1.000 | 0.800 | 0.889 | -0.058 |
| CONT_ENC | 0.933 | 0.933 | 0.933 | +0.183 |
| CONT_CON | 0.933 | 1.000 | 0.966 | +0.421 |

**Análisis de los 5 errores**: 2 son técnicos (`UnexpectedModelBehavior`, ids 30 y 57; eliminados del checkpoint para reintento al relanzar la celda). Los 4 reales comparten un único patrón: **el contenido social o la mención de subvenciones anula el acto contractual**:
- ids 23, 25: adjudicación de contratos de publicidad institucional que menciona "concesión de ayudas, subvenciones y convenios" → predice irrelevante.
- id 116: concierto sanitario MUGEJU con aseguradoras → lo cree subvención.
- id 110: contrato de servicios del programa de Termalismo del Imserso → lo cree programa social.

V3 ataca este patrón con la regla 8 ("la etiqueta la decide el ACTO, no el tema") y 3 ejemplos few-shot tomados de estos errores.

In [ ]:
agent_b4_v2 = build_agent(
    model, "v2",
    output_type=ClassifierOutputB4,
    prompt_registry=PROMPT_REGISTRY_B4,
)

df_b4_exp2 = await run_experiment(
    agent_b4_v2, df_anotado_run,
    use_n1_context=False, concurrency=1,
    output_path="../results/b4_exp2_promptv2_qwen9b.csv",
    desc="B4 Exp2 - Prompt v2",
)
df_b4_exp2.head(3)

In [ ]:
df_b4_exp2 = pd.read_csv("../results/b4_exp2_promptv2_qwen9b.csv")
df_eval_b4_2 = df_anotado[["id","is_relevant_gt","categories_gt","subcategories_gt","description"]].merge(
    df_b4_exp2[["description","is_relevant_pred","act_type_pred","categories_pred",
                "subcategories_pred","confidence","reasoning"]], on="description", how="left"
)
m_b4_2 = compute_metrics_B4(df_eval_b4_2, "Experimento B4-2 - Prompt v2")
print_errors_B4(df_eval_b4_2, m_b4_2["exact"], m_b4_2["rel"], label="B4 Experimento 2 - Prompt v2")

---

##  8. Experimento 3 - Prompt v3 (few-shot)

**Problema**: V2 (Macro F1=0.951) deja un único patrón real de error: el contenido social o la mención de subvenciones hace que el modelo descarte actos contractuales legítimos (publicidad institucional con mención de ayudas, concierto sanitario MUGEJU, termalismo Imserso). 4 errores, todos falsos negativos de relevancia.

**Objetivo**: Verificar si la regla 8 ("la etiqueta la decide el ACTO, no el tema") y 3 ejemplos few-shot tomados literalmente de los errores del Exp 2 eliminan el patrón sin degradar el resto.

**Enfoque**: Qwen 3.5 9B · SYSTEM_PROMPT_B4_V3 (V2 + regla 8 + 3 ejemplos, 6.164 chars) · few-shot · sin contexto N1.

**Resultados**: pendiente.

> Nota: reiniciar el kernel antes de ejecutar (el import de PROMPT_REGISTRY_B4 no se recarga solo). Ojo: los ejemplos 1-3 del prompt V3 son los ids 23, 116 y 110 del ground truth, por lo que el acierto en esos 3 registros no es generalización sino memorización; los ids 25, 102 y 112 (mismos patrones, no incluidos) son la validación honesta.

In [ ]:
agent_b4_v3 = build_agent(
    model, "v3",
    output_type=ClassifierOutputB4,
    prompt_registry=PROMPT_REGISTRY_B4,
)

df_b4_exp3 = await run_experiment(
    agent_b4_v3, df_anotado_run,
    use_n1_context=False, concurrency=1,
    output_path="../results/b4_exp3_promptv3_qwen9b.csv",
    desc="B4 Exp3 - Prompt v3",
)
df_b4_exp3.head(3)

In [ ]:
df_b4_exp3 = pd.read_csv("../results/b4_exp3_promptv3_qwen9b.csv")
df_eval_b4_3 = df_anotado[["id","is_relevant_gt","categories_gt","subcategories_gt","description"]].merge(
    df_b4_exp3[["description","is_relevant_pred","act_type_pred","categories_pred",
                "subcategories_pred","confidence","reasoning"]], on="description", how="left"
)
m_b4_3 = compute_metrics_B4(df_eval_b4_3, "Experimento B4-3 - Prompt v3")
print_errors_B4(df_eval_b4_3, m_b4_3["exact"], m_b4_3["rel"], label="B4 Experimento 3 - Prompt v3")

---

##  9. Comparativa de modelos - Gemma 4B

**Problema**: Todos los experimentos anteriores usan Qwen 3.5 9B. No sabemos si el prompt es transferible a modelos más pequeños y rápidos.

**Objetivo**: Comprobar si Gemma 4 4B con el mejor prompt compacto alcanza un rendimiento comparable al 9B.

**Enfoque**: Gemma 4 4B · mejor prompt que quepa en su ventana de contexto · zero-shot · sin contexto N1.

**Resultados**: pendiente.

In [ ]:
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider as OAIProvider

# Cargar gemma-4-e4b-it en LM Studio antes de ejecutar
model_gemma = OpenAIChatModel(
    "gemma-4-e4b-it",
    provider=OAIProvider(base_url="http://localhost:1234/v1", api_key="lm-studio")
)
agent_b4_gemma = build_agent(
    model_gemma, "v2",  # ajustar a la mejor versión compacta disponible
    output_type=ClassifierOutputB4,
    prompt_registry=PROMPT_REGISTRY_B4,
)

df_b4_gemma = await run_experiment(
    agent_b4_gemma, df_anotado_run,
    use_n1_context=False, concurrency=1,
    output_path="../results/b4_exp_gemma4b.csv",
    desc="B4 Gemma4B",
)
df_b4_gemma.head(3)

In [ ]:
df_b4_gemma = pd.read_csv("../results/b4_exp_gemma4b.csv")
df_eval_b4_gemma = df_anotado[["id","is_relevant_gt","categories_gt","subcategories_gt","description"]].merge(
    df_b4_gemma[["description","is_relevant_pred","act_type_pred","categories_pred",
                 "subcategories_pred","confidence","reasoning"]], on="description", how="left"
)
m_b4_gemma = compute_metrics_B4(df_eval_b4_gemma, "B4 Gemma 4B")
print_errors_B4(df_eval_b4_gemma, m_b4_gemma["exact"], m_b4_gemma["rel"], label="B4 Gemma 4B")

## 10. Tabla resumen - Comparativa de experimentos B4

In [ ]:
experimentos_cfg_B4 = [
    ("B4 Exp1 - Baseline V1", "Qwen 3.5 9B", "Zero-shot", "../results/b4_exp1_baseline_qwen9b.csv"),
    ("B4 Exp2 - Prompt V2",   "Qwen 3.5 9B", "Zero-shot", "../results/b4_exp2_promptv2_qwen9b.csv"),
    ("B4 Exp3 - Prompt V3",   "Qwen 3.5 9B", "Few-shot",  "../results/b4_exp3_promptv3_qwen9b.csv"),
    ("B4 Exp4 - Gemma 4B",    "Gemma 4 4B",  "Zero-shot", "../results/b4_exp_gemma4b.csv"),
]

rows_b4 = []
for nombre, modelo, config, path in experimentos_cfg_B4:
    if not Path(path).exists():
        print(f"  [SKIP] {nombre}: aún sin resultados ({path})")
        continue
    df_r = pd.read_csv(path)
    df_e = df_anotado[["id","is_relevant_gt","categories_gt","description"]].merge(
        df_r[["description","is_relevant_pred","categories_pred","confidence","duration_s"]],
        on="description", how="left"
    )
    m = compute_metrics_B4(df_e, verbose=False)
    rows_b4.append({
        "Experimento":     nombre,
        "Modelo":          modelo,
        "Config":          config,
        "is_rel_f1":       m["is_rel_f1"],
        "micro_f1":        m["micro_f1"],
        "macro_f1":        m["macro_f1"],
        "hamming_loss":    m["hamming_loss"],
        "jaccard_samples": m["jaccard_samples"],
        "subset_accuracy": m["subset_accuracy"],
        "mean_duration_s": round(df_r["duration_s"].mean(), 2) if "duration_s" in df_r.columns else None,
        "total_duration_s": round(df_r["duration_s"].sum(), 0) if "duration_s" in df_r.columns else None,
    })

df_summary_b4 = pd.DataFrame(rows_b4)
if not df_summary_b4.empty:
    df_display_b4 = df_summary_b4.copy()
    df_display_b4.columns = [
        "Experimento", "Modelo", "Config",
        "is_rel F1", "Micro F1", "Macro F1", "Hamming", "Jaccard", "Subset Acc",
        "s/item", "Total (s)",
    ]
    display(df_display_b4.set_index("Experimento"))